In [ ]:
import pandas as pd
import autoslo.utils.paths as pu

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

In [ ]:
df = pd.read_parquet(
    pu.get_redset_raw_data(cluster_type="provisioned", cluster_id=1)
)

In [98]:
df.head()

,instance_id,cluster_size,user_id,database_id,query_id,arrival_timestamp,compile_duration_ms,queue_duration_ms,execution_duration_ms,feature_fingerprint,was_aborted,was_cached,cache_source_query_id,query_type,num_permanent_tables_accessed,num_external_tables_accessed,num_system_tables_accessed,read_table_ids,write_table_ids,mbytes_scanned,mbytes_spilled,num_joins,num_scans,num_aggregations
0,1,7.0,1,0,1148627,2024-03-01 00:00:15.684430,2.0,0,947,5e54b9b80cc167e03534378bf76f845fb71f20528a076b8d44fa761ac6f9d1d0,0,0,NaN,select,1.0,0.0,0.0,1399,None,574.0,0.0,0,1,0
1,1,7.0,1,0,926707,2024-03-01 00:00:19.116287,793.0,0,814,ebee8fddae32b451626b3f52175764d9cd78c1fbbfb2b7b844ba2c82857f8374,0,0,NaN,ctas,62.0,0.0,0.0,"1286,1608,1609,1610,1611,1612,214,24,27,541",277238,769.0,0.0,10,11,7
2,1,7.0,1,0,949130,2024-03-01 00:00:19.459017,66.0,0,107,16852209e07054376af67f79d42f6b8c1cf0dd85fa6e454e7bda49a6a8422b59,0,0,NaN,select,0.0,0.0,4.0,None,None,0.0,0.0,3,0,0
3,1,7.0,1,0,1014260,2024-03-01 00:00:19.863207,192.0,0,218,315df4d5cf9c36971b14488e7dd82e0d53597d83f962e7937412ee9a88a06def,0,0,NaN,analyze,1.0,0.0,0.0,277238,None,378.0,0.0,0,1,2
4,1,7.0,1,0,979707,2024-03-01 00:00:21.985663,1774.0,0,1790,2f7237263308ca2aa91f9ebf8b42331438aff8e51e4e090ef69ec5cbc6837b57,0,0,NaN,ctas,36.0,0.0,0.0,"1608,1609,1611,1824,41,52,598",276633,642.0,0.0,7,8,11


In [ ]:
df["read_table_ids"].values[1]

'1286,1608,1609,1610,1611,1612,214,24,27,541'

In [103]:
df["query_hash"] = df.apply(
    lambda row: hash(row["feature_fingerprint"])
    + hash(row["query_type"])
    + hash(row["read_table_ids"])
    + hash(row["write_table_ids"])
    + hash(row["num_joins"])
    + hash(row["num_scans"])
    + hash(row["num_aggregations"]),
    axis=1,
)

In [104]:
df['query_hash'].value_counts()

query_hash
567793379063358307       7933
3308692791214937741      6086
-3786928993789567902     3903
10877695085400200271     3117
-2599224100294264169     2485
                         ... 
-5849103817289464703        1
4328184789562681496         1
-14793763738771353627       1
6767714393537206757         1
14834369048057829050        1
Name: count, Length: 945190, dtype: int64

In [106]:
df.groupby('query_hash')['execution_duration_ms'].var().sort_values(ascending=False).head(10)

query_hash
4124660608373612850     1.895687e+12
13234313116151523671    1.732297e+12
2362054419632256607     1.156782e+12
-1662977904482507086    8.530779e+11
11243568779339955427    6.775807e+11
-3747331804240975299    6.749245e+11
-3065856235676134901    6.714504e+11
8688437785048080839     6.664455e+11
9287696838157378282     6.625824e+11
-3734281576818899964    5.364615e+11
Name: execution_duration_ms, dtype: float64

In [119]:
# Filter only query hashes with at least 10 occurrences
df_frequent = df[df['query_type'].str.startswith('select')].groupby('query_hash').filter(lambda x: len(x) >= 10)

# Now calculate the variance for each query hash
df_frequent.groupby('query_hash')['execution_duration_ms'].var().sort_values(ascending=False).head(10)

query_hash
1820525319561951382      4.672793e+08
567793379063358307       3.765206e+08
-9920907603229185129     7.758014e+07
11121525973428380137     4.919760e+07
165292841079651529       2.455665e+07
488444142894608105       1.906670e+07
-5060764740871052828     1.285888e+07
-7388856811460795040     5.319974e+06
13123836642895263052     8.460890e+05
-12528690421801233675    8.053186e+05
Name: execution_duration_ms, dtype: float64

In [130]:
df.shape

(1251824, 25)

In [131]:
len(df[df['compile_duration_ms'] == 0])

8778

In [128]:
query_id = 11121525973428380137
df[df['query_hash'] == query_id  ]

,instance_id,cluster_size,user_id,database_id,query_id,arrival_timestamp,compile_duration_ms,queue_duration_ms,execution_duration_ms,feature_fingerprint,was_aborted,was_cached,cache_source_query_id,query_type,num_permanent_tables_accessed,num_external_tables_accessed,num_system_tables_accessed,read_table_ids,write_table_ids,mbytes_scanned,mbytes_spilled,num_joins,num_scans,num_aggregations,query_hash
1051389,1,7.0,0,0,1248741,2024-05-12 22:11:54.101299,160.0,0,3115,24e5f8b0d5ed720507339190d78ec05d053c9c9f1e856671d2b0ef14de4ab762,0,0,NaN,select,4.0,0.0,0.0,"25,288847",None,952.0,0.0,1,2,0,11121525973428380137
1051452,1,7.0,0,0,1197540,2024-05-12 22:14:17.855785,13373.0,0,17263,24e5f8b0d5ed720507339190d78ec05d053c9c9f1e856671d2b0ef14de4ab762,0,0,NaN,select,4.0,0.0,0.0,"25,288847",None,952.0,0.0,1,2,0,11121525973428380137
1051470,1,7.0,0,0,989425,2024-05-12 22:14:32.303299,233.0,0,4564,24e5f8b0d5ed720507339190d78ec05d053c9c9f1e856671d2b0ef14de4ab762,0,0,NaN,select,4.0,0.0,0.0,"25,288847",None,952.0,0.0,1,2,0,11121525973428380137
1051498,1,7.0,0,0,1056492,2024-05-12 22:15:15.199549,16799.0,0,21800,24e5f8b0d5ed720507339190d78ec05d053c9c9f1e856671d2b0ef14de4ab762,0,0,NaN,select,4.0,0.0,0.0,"25,288847",None,960.0,0.0,1,2,0,11121525973428380137
1051501,1,7.0,0,0,1120500,2024-05-12 22:15:32.917248,212.0,0,5473,24e5f8b0d5ed720507339190d78ec05d053c9c9f1e856671d2b0ef14de4ab762,0,0,NaN,select,4.0,0.0,0.0,"25,288847",None,964.0,0.0,1,2,0,11121525973428380137
1051504,1,7.0,0,0,953473,2024-05-12 22:16:10.580198,259.0,0,2511,24e5f8b0d5ed720507339190d78ec05d053c9c9f1e856671d2b0ef14de4ab762,0,0,NaN,select,4.0,0.0,0.0,"25,288847",None,952.0,0.0,1,2,0,11121525973428380137
1051505,1,7.0,0,0,1186706,2024-05-12 22:17:29.416921,159.0,0,2196,24e5f8b0d5ed720507339190d78ec05d053c9c9f1e856671d2b0ef14de4ab762,0,0,NaN,select,4.0,0.0,0.0,"25,288847",None,952.0,0.0,1,2,0,11121525973428380137
1051506,1,7.0,0,0,1158394,2024-05-12 22:17:37.084978,160.0,0,2185,24e5f8b0d5ed720507339190d78ec05d053c9c9f1e856671d2b0ef14de4ab762,0,0,NaN,select,4.0,0.0,0.0,"25,288847",None,952.0,0.0,1,2,0,11121525973428380137
1051509,1,7.0,0,0,1139919,2024-05-12 22:18:47.655000,205.0,0,3536,24e5f8b0d5ed720507339190d78ec05d053c9c9f1e856671d2b0ef14de4ab762,0,0,NaN,select,4.0,0.0,0.0,"25,288847",None,952.0,0.0,1,2,0,11121525973428380137
1051510,1,7.0,0,0,990892,2024-05-12 22:18:56.264938,186.0,0,2659,24e5f8b0d5ed720507339190d78ec05d053c9c9f1e856671d2b0ef14de4ab762,0,0,NaN,select,4.0,0.0,0.0,"25,288847",None,952.0,0.0,1,2,0,11121525973428380137


In [129]:
# How much of the variance is explained by compile_duration_ms?
sub_df = df[df['query_hash'] == query_id]
sub_df[['execution_duration_ms', 'compile_duration_ms']].corr()



,execution_duration_ms,compile_duration_ms
execution_duration_ms,1.000000,0.988705
compile_duration_ms,0.988705,1.000000


In [ ]:
df["execution_duration_ms"] = df["execution_duration_ms"].fillna(0)
df["queue_duration_ms"] = df["queue_duration_ms"].fillna(0)
df["compile_duration_ms"] = df["compile_duration_ms"].fillna(0)
df["end_timestamp"] = (
    df["arrival_timestamp"]
    + pd.to_timedelta(df["execution_duration_ms"], unit="ms")
    + pd.to_timedelta(df["queue_duration_ms"], unit="ms")
    + pd.to_timedelta(df["compile_duration_ms"], unit="ms")
)

In [ ]:
df = df.sort_values(by="arrival_timestamp")

# Find queries that execute in isolation (no other queries running at the same time)
isolated = []

events = []
for idx, row in df.iterrows():
    events.append((row["arrival_timestamp"], "1_start", row["query_id"]))
    events.append((row["end_timestamp"], "2_end", row["query_id"]))

events.sort()

running_queries = set()
saw_a_query_overlap = {}
for timestamp, event_type, query_id in events:
    if event_type == "1_start":
        if len(running_queries) > 0:
            saw_a_query_overlap[query_id] = True
            for rq in running_queries:
                saw_a_query_overlap[rq] = True
        running_queries.add(query_id)
    elif event_type == "2_end":
        running_queries.remove(query_id)

df["saw_a_query_overlap"] = df["query_id"].map(
    lambda qid: saw_a_query_overlap.get(qid, False)
)

In [ ]:
df["execution_minus_compile_duration_ms"] = (
    df["execution_duration_ms"] - df["compile_duration_ms"]
)

In [ ]:
df["execution_minus_compile_duration_ms"].describe()

count    1.251824e+06
mean     2.089132e+02
std      9.360352e+03
min     -8.084000e+03
25%      7.000000e+00
50%      1.600000e+01
75%      3.700000e+01
max      8.468890e+05
Name: execution_minus_compile_duration_ms, dtype: float64

In [ ]:
df.sort_values("execution_minus_compile_duration_ms").head()

,instance_id,cluster_size,user_id,database_id,query_id,arrival_timestamp,compile_duration_ms,queue_duration_ms,execution_duration_ms,feature_fingerprint,was_aborted,was_cached,cache_source_query_id,query_type,num_permanent_tables_accessed,num_external_tables_accessed,num_system_tables_accessed,read_table_ids,write_table_ids,mbytes_scanned,mbytes_spilled,num_joins,num_scans,num_aggregations,end_timestamp,saw_a_query_overlap,execution_minus_compile_duration_ms
732145,1,7.0,0,0,438563,2024-04-21 14:12:53.282705,20177.0,0,12093,8e760ee69973cb5635e09f73938204366c05cd90f6031d0dc1f156c64b38bced,0,0,NaN,ctas,751.0,0.0,0.0,"58,59",142154,170.0,0.0,1,2,4,2024-04-21 14:13:25.552705,True,-8084.0
954235,1,7.0,1,0,805407,2024-05-06 11:29:33.149379,17055.0,0,12115,None,1,0,NaN,select,1.0,0.0,0.0,47,None,40.0,0.0,0,1,1,2024-05-06 11:30:02.319379,True,-4940.0
402369,1,7.0,0,0,712960,2024-03-30 08:44:11.467782,0.0,49677,0,7a00688b6b6e88afcdad30b1d12527f07caf2ac13b4caf8d26b12b63808ae805,0,0,NaN,select,1134.0,0.0,0.0,None,None,NaN,NaN,0,0,0,2024-03-30 08:45:01.144782,True,0.0
119158,1,NaN,0,0,522028,2024-03-10 18:57:23.012623,0.0,0,0,None,0,1,592776.0,other,NaN,NaN,NaN,None,None,NaN,NaN,0,0,0,2024-03-10 18:57:23.012623,True,0.0
401627,1,NaN,0,0,504199,2024-03-30 08:09:51.376387,0.0,0,0,None,0,1,538890.0,other,NaN,NaN,NaN,None,None,NaN,NaN,0,0,0,2024-03-30 08:09:51.376387,True,0.0


In [ ]:
non_overlap_non_abort_df = df[
    ~df["saw_a_query_overlap"] & (df["was_aborted"] == False)
]
non_overlap_non_abort_df["execution_minus_compile_duration_ms"].describe()

count    15364.000000
mean        19.465439
std        194.051226
min          0.000000
25%          4.000000
50%          5.000000
75%          7.000000
max      10788.000000
Name: execution_minus_compile_duration_ms, dtype: float64

In [ ]:
non_overlap_non_abort_df.groupby("feature_fingerprint")[
    "execution_minus_compile_duration_ms"
].std().sort_values(ascending=False).head(10)

feature_fingerprint
cbcc4264f9b2fb6e9dea95655227d003bc40ce4575f8fe42f26769f92757ac80    1646.144587
586da31d4e5e186f9cd51a660844a7db01a381d8dcea3bf672dad8ef4f8885bb    1467.246571
9777732c37407e090aabe756b7bb893193fa2966fd651b3f28fb030e5892b320     248.268672
34a270b0db76da026409f43f5ce7f3d5d2e37f42d1a29c53b6704f89f65bf044     143.774128
9f4a2495634593b4bcccdbe8cfc9b9d31a58ea5ea190de55053c9b43354b4155     131.363047
31024b8ba7cad6d6577ff6512ec11c43ce25ad98a44f0bb79c8200445d30f753     123.036580
639034de5b5981442980c8aae6964dd1e67600f00e289ee5f3dc405366ab1493     119.176340
4a7094fa15d3cdd67978d4b57c80cb47e3a721594aa6b61f78cc0031799b7db9     118.793939
8fe4d248bf40f8f253fa3e227daa6011577ce00315eb735df26058e60d760994     111.722871
5fa094f2b549ad812dc4a46f183e4739f86d4612bd907931c86459a9c7c9e40b     106.462826
Name: execution_minus_compile_duration_ms, dtype: float64

In [ ]:
non_overlap_non_abort_df[
    non_overlap_non_abort_df["feature_fingerprint"]
    == "586da31d4e5e186f9cd51a660844a7db01a381d8dcea3bf672dad8ef4f8885bb"
]

,instance_id,cluster_size,user_id,database_id,query_id,arrival_timestamp,compile_duration_ms,queue_duration_ms,execution_duration_ms,feature_fingerprint,was_aborted,was_cached,cache_source_query_id,query_type,num_permanent_tables_accessed,num_external_tables_accessed,num_system_tables_accessed,read_table_ids,write_table_ids,mbytes_scanned,mbytes_spilled,num_joins,num_scans,num_aggregations,end_timestamp,saw_a_query_overlap,execution_minus_compile_duration_ms
1187422,1,7.0,0,0,290879,2024-05-24 10:54:44.128619,334.0,0,3066,586da31d4e5e186f9cd51a660844a7db01a381d8dcea3bf672dad8ef4f8885bb,0,0,NaN,select,4.0,0.0,0.0,"25,46710",None,952.0,0.0,1,2,0,2024-05-24 10:54:47.528619,False,2732.0
1187428,1,7.0,0,0,387866,2024-05-24 11:10:15.532290,405.0,0,5212,586da31d4e5e186f9cd51a660844a7db01a381d8dcea3bf672dad8ef4f8885bb,0,0,NaN,select,4.0,0.0,0.0,"25,46710",None,952.0,0.0,1,2,0,2024-05-24 11:10:21.149290,False,4807.0


In [ ]:
# Find the indexes of the events where the query with query_id 1053443 starts and ends
query_id_to_find = 907298
indexes = [
    i
    for i, (timestamp, event_type, query_id) in enumerate(events)
    if query_id == query_id_to_find
]
indexes

[13367, 13368]

In [37]:
l[0] < l[1]

False

In [ ]:
# Find which columns are funcitonally dependent on feature_fingerprint. That is,
# for each feature_fingerprint, there is exactly one value for that column, even
# though the column itself has multiple unique values across the whole dataframe.
dependent_columns = []
df_not_aborted = df[df["was_aborted"] == False]
for col in df_not_aborted.columns:
    if col == "feature_fingerprint":
        continue
    grouped = df_not_aborted.groupby("feature_fingerprint")[col].nunique()
    if (grouped <= 1).all() and df_not_aborted[col].nunique() > 1:
        dependent_columns.append(col)
dependent_columns

['was_cached', 'cache_source_query_id', 'num_system_tables_accessed']

In [ ]:
# Find a feature_fingerprint with multiple distinct values of num_joins and
# subset the dataframe to only rows with that fingerprint.
example_fingerprint = df_not_aborted.groupby("feature_fingerprint")[
    "num_joins"
].nunique()
example_fingerprint = example_fingerprint[example_fingerprint > 1].index[0]
df_subset = df_not_aborted[
    df_not_aborted["feature_fingerprint"] == example_fingerprint
]
df_subset

,instance_id,cluster_size,user_id,database_id,query_id,arrival_timestamp,compile_duration_ms,queue_duration_ms,execution_duration_ms,feature_fingerprint,was_aborted,was_cached,cache_source_query_id,query_type,num_permanent_tables_accessed,num_external_tables_accessed,num_system_tables_accessed,read_table_ids,write_table_ids,mbytes_scanned,mbytes_spilled,num_joins,num_scans,num_aggregations
389441,1,7.0,0,0,1147330,2024-03-29 11:57:19.547849,7003.0,0,7162,3da19094df6f0e47ca30cb22b3d5835d46277183c034be27d177c94a05576b57,0,0,NaN,select,665.0,0.0,0.0,"10,12,143,16,17,19,2,22,221,222,23,25,26,28,29,30,31,32,34,35,36,37,38,39,40,42,43,47,48,49,53,54,55,57,9",None,4762.0,0.0,54,53,64
389565,1,7.0,0,0,970304,2024-03-29 12:10:31.628287,NaN,54754,0,3da19094df6f0e47ca30cb22b3d5835d46277183c034be27d177c94a05576b57,0,0,NaN,select,665.0,0.0,0.0,None,None,NaN,NaN,0,0,0
389568,1,7.0,0,0,1028620,2024-03-29 12:11:42.748011,NaN,51347,0,3da19094df6f0e47ca30cb22b3d5835d46277183c034be27d177c94a05576b57,0,0,NaN,select,665.0,0.0,0.0,None,None,NaN,NaN,0,0,0
389587,1,7.0,0,0,1197478,2024-03-29 12:13:50.002297,6555.0,0,6691,3da19094df6f0e47ca30cb22b3d5835d46277183c034be27d177c94a05576b57,0,0,NaN,select,665.0,0.0,0.0,"10,12,143,16,17,19,2,22,221,222,23,25,26,28,29,30,31,32,34,35,36,37,38,39,40,42,43,47,48,49,53,54,55,57,9",None,4762.0,0.0,54,53,64
389696,1,7.0,0,0,988345,2024-03-29 13:09:32.840180,NaN,57789,0,3da19094df6f0e47ca30cb22b3d5835d46277183c034be27d177c94a05576b57,0,0,NaN,select,665.0,0.0,0.0,None,None,NaN,NaN,0,0,0
389698,1,7.0,0,0,1222678,2024-03-29 13:10:36.094009,NaN,57941,0,3da19094df6f0e47ca30cb22b3d5835d46277183c034be27d177c94a05576b57,0,0,NaN,select,665.0,0.0,0.0,None,None,NaN,NaN,0,0,0
